# S18 · Calidad del aire en México · Notebook 02: reporte PDF

| | |
|---|---|
| **Programa** | IA Explorer – TCS / TecNM (CPFIA) |
| **Módulo** | 02 |
| **Sesión** | 18 · Excel para análisis de datos I |
| **Fecha** | 2026-11-12 |
| **Autor** | Luis Saúl González Guerra |

## 1. Objetivo

Generar el dashboard en PDF que muestra el proceso completo, de los datos crudos a la decisión, en el mismo orden en que se realizó:

1. Portada con la pregunta de negocio y el Excel Mindset.
2. Etapa 1: el archivo original de la OMS abierto en Excel.
3. Etapas 1 y 2 en Python: el notebook `01_preparacion_datos.ipynb` completo, con su código y resultados.
4. Etapas 2 a 8 en Excel: una captura de pantalla por paso, tomada durante el trabajo en `outputs/S18_Calidad_Aire_Mexico.xlsx`.

Las capturas se guardaron en `outputs/capturas/` con un nombre fijo por paso. Si alguna no existe, su página se marca en rojo.

## 2. Configuración

In [1]:
import shutil
import subprocess
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import nbformat
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from nbconvert import HTMLExporter
from pypdf import PdfReader, PdfWriter

BASE_DIR = Path.cwd().parent
CAPTURAS_DIR = BASE_DIR / "outputs" / "capturas"
OUTPUT_DIR = BASE_DIR / "outputs"
CAPTURAS_DIR.mkdir(parents=True, exist_ok=True)
PDF_OUT = OUTPUT_DIR / "S18_Dashboard_Calidad_Aire_Mexico.pdf"
NOTEBOOK_01 = BASE_DIR / "notebooks" / "01_preparacion_datos.ipynb"
CHROME_RUTAS = [
    "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
    "C:/Program Files/Google/Chrome/Application/chrome.exe",
    shutil.which("google-chrome") or "",
    shutil.which("chromium") or "",
]

AUTOR = "Luis Saúl González Guerra"
FECHA = "Noviembre 2026"
A4 = (11.69, 8.27)
AZUL, NARANJA, GRIS, TINTA = "#2F5D8A", "#E8704A", "#9AA5B1", "#1F2A36"

## 3. Capturas del proceso

Cada página del reporte corresponde a una captura. La primera va antes del notebook 01 y el resto después.

In [2]:
CAPTURA_INICIAL = ("Etapa 1 · Datos crudos", "Archivo original de la OMS abierto en Excel", "01_datos_crudos")
CAPTURAS = [
    ("Etapa 2 · Datos limpios", "Datos limpios abiertos en Excel", "02_datos_limpios"),
    ("Etapa 3 · Tabla de Excel", "Rango convertido en la Tabla tblCalidadAire", "03_tabla_excel"),
    ("Etapa 4 · Fórmulas y cálculos", "Columnas calculadas con SI", "04_columnas_calculadas"),
    ("Etapa 4 · Fórmulas y cálculos", "Fórmulas base como utilidad", "05_formulas_base"),
    ("Etapa 5 · Tabla dinámica", "Mapeo de la pregunta a Filas, Columnas, Valores y Filtros", "06_tabla_dinamica_campos"),
    ("Etapa 5 · Tabla dinámica", "PM2.5 promedio por entidad y año", "07_tabla_dinamica"),
    ("Etapa 6 · Gráficos", "Gráficos dinámicos", "08_grafico_dinamico"),
    ("Etapa 7 · Dashboard", "Segmentadores y conexiones de informe", "09_segmentadores"),
    ("Etapa 7 · Dashboard", "Dashboard de calidad del aire", "10_dashboard"),
    ("Etapa 7 · Dashboard", "Dashboard filtrado con los segmentadores", "11_dashboard_filtrado"),
    ("Etapa 8 · Reporte de negocio", "Insight y decisión", "12_decision"),
]

def buscar_captura(nombre):
    for ext in (".png", ".jpg", ".jpeg"):
        ruta = CAPTURAS_DIR / f"{nombre}{ext}"
        if ruta.exists():
            return ruta
    return None

estado = pd.DataFrame([(clave, titulo, "Sí" if buscar_captura(clave) else "Falta")
                       for _, titulo, clave in [CAPTURA_INICIAL] + CAPTURAS],
                      columns=["Archivo", "Contenido", "Captura"])
estado

,Archivo,Contenido,Captura
0,01_datos_crudos,Archivo original de la OMS abierto en Excel,Sí
1,02_datos_limpios,Datos limpios abiertos en Excel,Sí
2,03_tabla_excel,Rango convertido en la Tabla tblCalidadAire,Sí
3,04_columnas_calculadas,Columnas calculadas con SI,Sí
4,05_formulas_base,Fórmulas base como utilidad,Sí
5,06_tabla_dinamica_campos,"Mapeo de la pregunta a Filas, Columnas, Valore...",Sí
6,07_tabla_dinamica,PM2.5 promedio por entidad y año,Sí
7,08_grafico_dinamico,Gráficos dinámicos,Sí
8,09_segmentadores,Segmentadores y conexiones de informe,Sí
9,10_dashboard,Dashboard de calidad del aire,Sí


## 4. Notebook 01 a PDF

El notebook 01 se convierte a HTML con `nbconvert` y Google Chrome lo imprime a PDF sin abrir ventana. Se convierte primero para conocer su número de páginas y numerar de forma continua el resto del reporte.

In [3]:
chrome = next((r for r in CHROME_RUTAS if r and Path(r).exists()), None)
if chrome is None:
    raise FileNotFoundError("No se encontró Google Chrome. Se puede agregar su ruta a CHROME_RUTAS.")

nb01 = nbformat.read(NOTEBOOK_01, as_version=4)
html, _ = HTMLExporter(template_name="lab").from_notebook_node(nb01)
estilo = ("<style>@page{size:A4;margin:12mm} body{font-size:11px}"
          " .jp-OutputArea-output pre, .jp-InputArea-editor pre{white-space:pre-wrap;word-break:break-word}"
          " .jp-RenderedHTMLCommon table{font-size:8px}</style></head>")
NOTEBOOK_HTML = OUTPUT_DIR / "_notebook01.html"
NOTEBOOK_PDF = OUTPUT_DIR / "_notebook01.pdf"
NOTEBOOK_HTML.write_text(html.replace("</head>", estilo, 1), encoding="utf-8")

subprocess.run([chrome, "--headless=new", "--disable-gpu", "--no-pdf-header-footer",
                f"--print-to-pdf={NOTEBOOK_PDF}", NOTEBOOK_HTML.as_uri()],
               check=True, capture_output=True, timeout=180)
paginas_notebook = len(PdfReader(NOTEBOOK_PDF).pages)
print(f"Notebook 01 convertido: {paginas_notebook} páginas")

Notebook 01 convertido: 13 páginas


## 5. Generación del PDF

El reporte se arma en dos partes, antes y después del notebook, y al final se unen las tres en un solo archivo.

In [4]:
def nueva_pagina(pdf, etapa, titulo, inicio=0):
    fig = plt.figure(figsize=A4)
    if etapa:
        fig.text(0.05, 0.945, etapa.upper(), fontsize=9, color=NARANJA, fontweight="bold")
    fig.text(0.05, 0.9, titulo, fontsize=19, fontweight="bold", color=TINTA)
    fig.text(0.05, 0.03, f"Práctica S18 · IA Explorer TCS / TecNM (CPFIA) · {AUTOR}", fontsize=8, color=GRIS)
    fig.text(0.95, 0.03, str(inicio + pdf.get_pagecount() + 1), fontsize=8, color=GRIS, ha="right")
    return fig

def pagina_captura(pdf, etapa, titulo, clave, inicio=0):
    fig = nueva_pagina(pdf, etapa, titulo, inicio)
    ax = fig.add_axes([0.05, 0.07, 0.9, 0.79])
    ax.set_xticks([]), ax.set_yticks([])
    ruta = buscar_captura(clave)
    if ruta is None:
        ax.set_facecolor("#F4F6F8")
        ax.text(0.5, 0.5, f"Captura pendiente:\noutputs/capturas/{clave}.png", ha="center", va="center",
                fontsize=12, color="red", transform=ax.transAxes)
    else:
        ax.imshow(mpimg.imread(ruta))
        ax.set_aspect("equal")
        for s in ax.spines.values():
            s.set_edgecolor("#C9D3DD")
    pdf.savefig(fig)
    plt.close(fig)

In [5]:
PARTE_A = OUTPUT_DIR / "_parte_a.pdf"
PARTE_B = OUTPUT_DIR / "_parte_b.pdf"

with PdfPages(PARTE_A) as pdf:
    fig = nueva_pagina(pdf, None, "Calidad del aire en México: de datos crudos a decisiones")
    fig.text(0.05, 0.86, f"Práctica 1 · Módulo 2 · Sesión 18, Excel para análisis de datos I · {AUTOR} · {FECHA}",
             fontsize=10.5, color="#5A6773", style="italic")
    fig.text(0.05, 0.62, "Excel Mindset: la herramienta es el vehículo, la pregunta es el destino",
             fontsize=12, fontweight="bold", color=TINTA)
    ax = fig.add_axes([0.05, 0.38, 0.9, 0.2])
    ax.axis("off")
    mindset = [["¿Qué entidades de México\nrespiran el aire más contaminado\ny cómo ha cambiado?",
                "Promedio anual de\nPM2.5 (µg/m³)", "Entidad (filas)\ny año (columnas)", "Región y año",
                "Priorizar dónde invertir primero\nen control de emisiones y monitoreo"]]
    tabla = ax.table(cellText=mindset, colLabels=["Pregunta", "Métrica", "Dimensión", "Filtro", "Análisis accionable"],
                     colWidths=[0.28, 0.15, 0.15, 0.12, 0.3], bbox=[0, 0, 1, 1], cellLoc="left")
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9.5)
    for (r, _), celda in tabla.get_celld().items():
        celda.set_edgecolor("#C9D3DD")
        if r == 0:
            celda.set_facecolor(AZUL)
            celda.get_text().set_color("white")
            celda.get_text().set_fontweight("bold")
    fig.text(0.05, 0.3, "Fuente: WHO Ambient Air Quality Database v8.0 (2026); mediciones de México del SINAICA, INECC.",
             fontsize=9, color="#5A6773")
    fig.text(0.05, 0.27, "Referencias: WHO Global Air Quality Guidelines (2021) · Material de la Sesión 18, "
             "TCS Empowers / TecNM CPFIA.", fontsize=9, color="#5A6773")
    pdf.savefig(fig)
    plt.close(fig)

    pagina_captura(pdf, *CAPTURA_INICIAL)
    paginas_a = pdf.get_pagecount()

inicio_b = paginas_a + paginas_notebook
with PdfPages(PARTE_B) as pdf:
    for etapa, titulo, clave in CAPTURAS:
        pagina_captura(pdf, etapa, titulo, clave, inicio_b)

writer = PdfWriter()
for parte in (PARTE_A, NOTEBOOK_PDF, PARTE_B):
    writer.append(str(parte))
with open(PDF_OUT, "wb") as f:
    writer.write(f)
writer.close()
for temporal in (PARTE_A, PARTE_B, NOTEBOOK_PDF, NOTEBOOK_HTML):
    temporal.unlink()

faltan = int((estado["Captura"] == "Falta").sum())
print(f"PDF guardado: {PDF_OUT.relative_to(BASE_DIR)} ({len(PdfReader(PDF_OUT).pages)} páginas)")
print("Todas las capturas están incluidas." if faltan == 0 else f"Capturas pendientes: {faltan}")

PDF guardado: outputs/S18_Dashboard_Calidad_Aire_Mexico.pdf (26 páginas)
Todas las capturas están incluidas.
